# 🌏 Savannakhet GNN — Real Data Training
**ใช้ข้อมูลจริงจากฐานข้อมูล Savannakhet**

## 📋 ขั้นตอนก่อนรัน Notebook นี้
1. รันสคริปต์ที่เครื่องตัวเองก่อน:
   ```bash
   cd c:/Users/ASUS/savannakhet-project/backend
   python scripts/export_data_for_colab.py
   ```
2. อัปโหลดไฟล์ **`savannakhet_real_data.json`** ขึ้น Colab (Cell ถัดไปจะช่วยทำ)
3. รัน Cell ตามลำดับ ✅

---
```
ข้อมูลจริง (JSON) ──► HeteroGraph ──► HeteroGraphSAGE (2 Layers)
                                   ──► GNN Score (60%) + Content Score (40%)
                                   ──► Hybrid Fusion ──► Top-K Recommendations
```

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1: ติดตั้ง PyTorch Geometric                       ║
# ╚══════════════════════════════════════════════════════════╝
import subprocess, sys, torch

print('⏳ ติดตั้ง torch_geometric...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch_geometric', '-q'], check=True)

torch_ver = torch.__version__.split('+')[0]
cuda_tag  = 'cu121' if torch.cuda.is_available() else 'cpu'
for pkg in ['torch_scatter', 'torch_sparse', 'torch_cluster']:
    try:
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', pkg, '-q',
            '-f', f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
        ], check=True)
    except Exception as e:
        print(f'⚠️  {pkg}: {e}')

print('✅ ติดตั้งเสร็จ!')
print(f'   PyTorch : {torch.__version__}')
print(f'   CUDA    : {torch.cuda.is_available()}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2: Upload ไฟล์ข้อมูลจริง                          ║
# ╚══════════════════════════════════════════════════════════╝
from google.colab import files
import json, os

print('📂 กรุณาเลือกไฟล์  savannakhet_real_data.json')
print('   (ไฟล์นี้อยู่ที่ backend/docs/savannakhet_real_data.json)')
print()

uploaded = files.upload()

# โหลดข้อมูล
json_filename = list(uploaded.keys())[0]
with open(json_filename, 'r', encoding='utf-8') as f:
    DB = json.load(f)

CATEGORIES   = DB['categories']
USERS_DATA   = DB['users']
PLACES_DATA  = DB['places']
INTERACTIONS = DB['interactions']
stats        = DB['stats']

print(f'✅ โหลดข้อมูลจริงสำเร็จ!')
print(f'   👤 Users        : {stats["num_users"]}')
print(f'   📍 Places       : {stats["num_places"]}')
print(f'   🔗 Interactions : {stats["num_interactions"]}')
print(f'   🏷️  Categories   : {len(CATEGORIES)}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3: Import & Setup                                  ║
# ╚══════════════════════════════════════════════════════════╝
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, to_hetero
from torch_geometric.data import HeteroData
from torch_geometric.utils import negative_sampling
import matplotlib.pyplot as plt
import numpy as np
import time, random

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ พร้อมแล้ว  |  Device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4: สร้าง HeteroGraph จากข้อมูลจริง                ║
# ╚══════════════════════════════════════════════════════════╝

# สร้าง index mapping: DB id → tensor index
user_id2idx  = {u['id']: i for i, u in enumerate(USERS_DATA)}
place_id2idx = {p['id']: i for i, p in enumerate(PLACES_DATA)}
place_idx2id = {i: p['id'] for i, p in enumerate(PLACES_DATA)}

NUM_USERS  = len(USERS_DATA)
NUM_PLACES = len(PLACES_DATA)

def build_real_graph():
    data = HeteroData()

    # ── User Features: Multi-hot (10 dim) ──
    user_feats = []
    for u in USERS_DATA:
        prefs = u.get('preferences', [])
        vec = [1.0 if cat in prefs else 0.0 for cat in CATEGORIES]
        user_feats.append(vec)
    data['user'].x = torch.tensor(user_feats, dtype=torch.float)

    # ── Place Features: One-hot category (10 dim) + rating (1 dim) = 11 dim ──
    place_feats = []
    for p in PLACES_DATA:
        vec = [1.0 if cat == p['category'] else 0.0 for cat in CATEGORIES]
        vec.append(min(p['rating'] / 5.0, 1.0))  # normalize
        place_feats.append(vec)
    data['place'].x = torch.tensor(place_feats, dtype=torch.float)

    # ── Edges จาก Interactions จริง ──
    # น้ำหนัก: view=1, like=2, review=3
    WEIGHT_MAP = {'view': 1.0, 'like': 2.0, 'review': 3.0}
    edge_src, edge_dst = [], []
    added = set()

    for intr in INTERACTIONS:
        uid = intr['user_id']
        pid = intr['place_id']
        if uid not in user_id2idx or pid not in place_id2idx:
            continue  # ข้ามถ้า user/place ถูกลบแล้ว
        u_idx = user_id2idx[uid]
        p_idx = place_id2idx[pid]
        key = (u_idx, p_idx)
        if key not in added:
            added.add(key)
            edge_src.append(u_idx)
            edge_dst.append(p_idx)

    # ── Fallback: ถ้า interactions น้อยเกินไป เพิ่มจาก preference ──
    if len(edge_src) < NUM_USERS * 2:
        print(f'⚠️  Interactions น้อย ({len(edge_src)}) — เพิ่ม preference-based edges')
        for u_idx, u in enumerate(USERS_DATA):
            prefs = u.get('preferences', [])
            for p_idx, p in enumerate(PLACES_DATA):
                if p['category'] in prefs and (u_idx, p_idx) not in added:
                    added.add((u_idx, p_idx))
                    edge_src.append(u_idx)
                    edge_dst.append(p_idx)

    data['user', 'interacts_with', 'place'].edge_index = torch.tensor(
        [edge_src, edge_dst], dtype=torch.long
    )

    return data

data = build_real_graph().to(device)

print(f'✅ สร้าง Graph จากข้อมูลจริงสำเร็จ!')
print(f'   User nodes  : {data["user"].x.shape}   — {NUM_USERS} คน × 10 features')
print(f'   Place nodes : {data["place"].x.shape}  — {NUM_PLACES} แห่ง × 11 features')
print(f'   Edges       : {data["user", "interacts_with", "place"].edge_index.shape[1]} connections')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5: นิยาม HeteroGraphSAGE Model                    ║
# ╚══════════════════════════════════════════════════════════╝

class BaseGNN(torch.nn.Module):
    def __init__(self, hidden_channels=64):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), hidden_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

model     = to_hetero(BaseGNN(hidden_channels=64), metadata=data.metadata()).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

total_params = sum(p.numel() for p in model.parameters())
print(f'✅ Model พร้อม!')
print(f'   Architecture : HeteroGraphSAGE (K=2, hidden=64)')
print(f'   Parameters   : {total_params:,}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6: 🚀 เทรนโมเดลด้วยข้อมูลจริง (100 Epochs)       ║
# ╚══════════════════════════════════════════════════════════╝

EPOCHS = 100
losses, accuracies = [], []
pos_edge_index = data['user', 'interacts_with', 'place'].edge_index
n_steps = max(1, pos_edge_index.size(1) // 5)

print('=' * 65)
print('  🚀  เทรน HeteroGraphSAGE ด้วยข้อมูลจริง (100 Epochs)')
print(f'  📊  {NUM_USERS} users | {NUM_PLACES} places | {pos_edge_index.shape[1]} edges')
print('=' * 65)

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    optimizer.zero_grad()

    out = model(data.x_dict, data.edge_index_dict)

    ps, pd  = pos_edge_index[0], pos_edge_index[1]
    pos_out = (out['user'][ps] * out['place'][pd]).sum(dim=-1)

    neg_ei  = negative_sampling(
        edge_index=pos_edge_index,
        num_nodes=(data['user'].num_nodes, data['place'].num_nodes),
        num_neg_samples=pos_edge_index.size(1)
    )
    ns, nd  = neg_ei[0], neg_ei[1]
    neg_out = (out['user'][ns] * out['place'][nd]).sum(dim=-1)

    labels  = torch.cat([torch.ones(pos_out.size(0)), torch.zeros(neg_out.size(0))]).to(device)
    loss    = F.binary_cross_entropy_with_logits(torch.cat([pos_out, neg_out]), labels)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    with torch.no_grad():
        correct = (pos_out >= 0).float().sum() + (neg_out < 0).float().sum()
        acc = correct.item() / (pos_out.size(0) + neg_out.size(0))
        accuracies.append(acc)

    if epoch % 5 == 0 or epoch == 1:
        ms = int((time.time() - t0) * 1000 / max(1, n_steps))
        print(f'Epoch {epoch:>3}/{EPOCHS}  '
              f'{n_steps}/{n_steps} ━━━━━━━━━━━━━━━━━━━━  '
              f'0s {ms}ms/step  '
              f'accuracy: {acc:.4f}  loss: {loss.item():.4f}')

print(f'\n✅ เทรนเสร็จ!  Accuracy: {accuracies[-1]*100:.2f}%  |  Loss: {losses[-1]:.4f}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7: 📊 กราฟ Training Results                       ║
# ╚══════════════════════════════════════════════════════════╝

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Savannakhet GNN — Real Data Training ({NUM_USERS} users | {NUM_PLACES} places)',
             fontsize=12, fontweight='bold')

ep = range(1, EPOCHS + 1)

ax1.plot(ep, accuracies, color='#2563EB', lw=2, label='Accuracy')
ax1.fill_between(ep, accuracies, alpha=0.15, color='#2563EB')
ax1.axhline(0.85, color='#059669', ls='--', lw=1.5, label='Target 85%')
ax1.set(title='Model Accuracy', xlabel='Epochs', ylabel='Accuracy', ylim=(0.4, 1.05))
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}%'))
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep, losses, color='#DC2626', lw=2, label='Loss')
ax2.fill_between(ep, losses, alpha=0.12, color='#DC2626')
ax2.set(title='Model Loss', xlabel='Epochs', ylabel='Loss', ylim=(0, None))
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/real_training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ บันทึกกราฟ: /content/real_training_results.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8: 🌟 ทดสอบ Recommendation ด้วยข้อมูลจริง        ║
# ╚══════════════════════════════════════════════════════════╝

def recommend_real(username: str, top_k: int = 5):
    model.eval()

    # หา user
    user_rec = next((u for u in USERS_DATA if u['username'] == username), None)
    if user_rec is None:
        print(f'❌ ไม่พบ user: {username}')
        print(f'   Users ที่มี: {[u["username"] for u in USERS_DATA[:10]]}')
        return

    u_idx = user_id2idx[user_rec['id']]
    prefs = user_rec.get('preferences', [])

    with torch.no_grad():
        embs   = model(data.x_dict, data.edge_index_dict)
        u_emb  = embs['user'][u_idx]
        p_embs = embs['place']

        # GNN Score
        gnn_scores = F.cosine_similarity(p_embs, u_emb.unsqueeze(0))

        # Content-Based Score
        u_pref_vec = data['user'].x[u_idx, :len(CATEGORIES)]
        p_cat_vecs = data['place'].x[:, :len(CATEGORIES)]
        cb_scores = (F.cosine_similarity(p_cat_vecs, u_pref_vec.unsqueeze(0))
                     if u_pref_vec.sum() > 0 else torch.zeros(NUM_PLACES).to(device))

        # Hybrid Score
        scores = 0.6 * gnn_scores + 0.4 * cb_scores
        top_vals, top_idxs = torch.topk(scores, k=min(top_k, NUM_PLACES))

    print('=' * 65)
    print(f'  🌟 Top {top_k} คำแนะนำสำหรับ: {username}')
    print(f'  🎯 ความชอบ: {prefs if prefs else "(ไม่ได้ตั้งค่า)"}')
    print('=' * 65)
    for rank, (val, idx) in enumerate(zip(top_vals, top_idxs), 1):
        p    = PLACES_DATA[idx.item()]
        pct  = max(0, min(100, (val.item() + 1) / 2 * 100))
        gnn  = max(0, min(100, (gnn_scores[idx].item() + 1) / 2 * 100))
        cb   = max(0, min(100, (cb_scores[idx].item() + 1) / 2 * 100))
        mark = '✓' if p['category'] in prefs else ' '
        print(f'  {rank}. {mark} {p["name"]:<30} [{p["category"]:<12}] ⭐{p["rating"]:.1f}')
        print(f'       💯 {pct:.1f}%  │  👥 GNN: {gnn:.1f}%  │  🎯 CB: {cb:.1f}%')
    print()

# ── ทดสอบกับ user จริง ──
# เปลี่ยนชื่อ username ให้ตรงกับ database ของคุณ
test_users = [u['username'] for u in USERS_DATA[:3]]  # ใช้ 3 คนแรก
for username in test_users:
    recommend_real(username, top_k=5)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 9: 💾 บันทึกโมเดล + Download                      ║
# ╚══════════════════════════════════════════════════════════╝

SAVE_PATH = '/content/gnn_savannakhet_real.pt'
torch.save({
    'epoch'        : EPOCHS,
    'model_state'  : model.state_dict(),
    'final_acc'    : accuracies[-1],
    'final_loss'   : losses[-1],
    'user_id2idx'  : user_id2idx,
    'place_id2idx' : place_id2idx,
    'place_idx2id' : place_idx2id,
    'metadata': {
        'hidden_dim' : 64,
        'categories' : CATEGORIES,
        'num_users'  : NUM_USERS,
        'num_places' : NUM_PLACES,
        'source'     : 'real_database',
    }
}, SAVE_PATH)

print('\n' + '═' * 65)
print('       📊  สรุปผลการเทรนด้วยข้อมูลจริง')
print('═' * 65)
print(f'  Architecture : HeteroGraphSAGE (K=2, hidden=64)')
print(f'  Dataset      : {NUM_USERS} users × {NUM_PLACES} places (Real DB)')
print(f'  Edges        : {data["user", "interacts_with", "place"].edge_index.shape[1]}')
print(f'  Epochs       : {EPOCHS}')
print(f'  Best Acc     : {max(accuracies)*100:.2f}%')
print(f'  Final Acc    : {accuracies[-1]*100:.2f}%')
print(f'  Final Loss   : {losses[-1]:.4f}')
print('═' * 65)

# Download
from google.colab import files
files.download(SAVE_PATH)
files.download('/content/real_training_results.png')
print('\n✅ กำลัง Download...')
print('   📦 gnn_savannakhet_real.pt    ← โมเดลที่เทรนแล้ว')
print('   📊 real_training_results.png  ← กราฟผลการเทรน')
print()
print('👉 นำไฟล์ .pt ไปวางที่ backend/gnn_model.pt เพื่อใช้ใน Production!')